# Hand Gesture CNN (Simple, Colab)

- Colab GPU 환경에서 실행
- `kaggle.json` 업로드 후 Kaggle API로 LeapGestRecog 다운로드
- 기본 CNN 모델로 학습/평가

In [ ]:
# 1) Colab 환경 확인 + 필수 패키지 설치
!pip -q install kaggle

try:
    import google.colab  # noqa: F401
except ImportError:
    raise EnvironmentError('이 노트북은 Google Colab에서 실행하세요.')

import os
import zipfile
import shutil
import subprocess
import random
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from google.colab import files
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report

if not torch.cuda.is_available():
    raise RuntimeError('GPU가 꺼져 있습니다. Runtime > Change runtime type > GPU로 변경하세요.')

device = torch.device('cuda')
print('Device:', device)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) kaggle.json 업로드 + 데이터셋 다운로드/압축해제
KAGGLE_DATASET = 'gti-upm/leapgestrecog'  # 필요 시 수정
DOWNLOAD_DIR = '/content/kaggle_data'
UNZIP_DIR = '/content'

print('kaggle.json 파일을 업로드하세요.')
uploaded = files.upload()
if 'kaggle.json' not in uploaded:
    raise FileNotFoundError('kaggle.json 업로드가 필요합니다.')

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
cmd = [
    'kaggle', 'datasets', 'download',
    '-d', KAGGLE_DATASET,
    '-p', DOWNLOAD_DIR,
    '--force'
]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Kaggle 다운로드 실패: slug 또는 kaggle.json을 확인하세요.')

zip_files = [f for f in os.listdir(DOWNLOAD_DIR) if f.endswith('.zip')]
if not zip_files:
    raise FileNotFoundError('다운로드된 zip 파일이 없습니다.')

for z in zip_files:
    zpath = os.path.join(DOWNLOAD_DIR, z)
    with zipfile.ZipFile(zpath, 'r') as zf:
        zf.extractall(UNZIP_DIR)
    print('Unzipped:', zpath)

BASE_PATH_CANDIDATES = [
    '/content/leapGestRecog',
    '/content/gesture/leapGestRecog',
]

BASE_PATH = None
for p in BASE_PATH_CANDIDATES:
    if os.path.isdir(p):
        BASE_PATH = p
        break

if BASE_PATH is None:
    raise FileNotFoundError('leapGestRecog 폴더를 찾지 못했습니다.')

print('BASE_PATH:', BASE_PATH)

In [ ]:
# 3) 데이터셋/모델/학습
def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

TRAIN_SUBJECTS = ['00', '01', '02', '03', '04', '05']
VAL_SUBJECTS = ['06', '07']
TEST_SUBJECTS = ['08', '09']

INPUT_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

class LeapGestRecogDataset(Dataset):
    CLASSES = sorted([
        '01_palm', '02_l', '03_fist', '04_fist_moved', '05_thumb',
        '06_index', '07_ok', '08_palm_moved', '09_c', '10_down'
    ])

    def __init__(self, base_path, subjects, transform=None):
        self.transform = transform
        self.classes = self.CLASSES
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = []
        for subj in subjects:
            for cls in self.classes:
                cls_path = os.path.join(base_path, subj, cls)
                if not os.path.isdir(cls_path):
                    continue
                for fname in os.listdir(cls_path):
                    path = os.path.join(cls_path, fname)
                    if os.path.isfile(path):
                        self.samples.append((path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
])
eval_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
])

train_dataset = LeapGestRecogDataset(BASE_PATH, TRAIN_SUBJECTS, train_tf)
val_dataset = LeapGestRecogDataset(BASE_PATH, VAL_SUBJECTS, eval_tf)
test_dataset = LeapGestRecogDataset(BASE_PATH, TEST_SUBJECTS, eval_tf)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('Train:', len(train_dataset), 'Val:', len(val_dataset), 'Test:', len(test_dataset))

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN(num_classes=len(train_dataset.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

best_val_acc = 0.0
best_state = None

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss {tr_loss:.4f} Acc {tr_acc:.2f}% | Val Loss {va_loss:.4f} Acc {va_acc:.2f}%')
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), '/content/best_hand_gesture_cnn.pt')
print('Best Val Acc:', round(best_val_acc, 2), '%')
print('Saved: /content/best_hand_gesture_cnn.pt')

In [ ]:
# 4) 테스트 평가
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=train_dataset.classes, digits=4))